# LSTM-Autoencoder Training on Google Colab (T4 GPU)
## Cold Chain EWS Digital Twin — Milestone 4a / 4b

---

> **⚠ STRUCTURAL PROXY NOTICE**  
> All data in this notebook derives from `IOT-temp.csv` — a **generic IoT temperature sensor log,  
> NOT real refrigerated-transport telemetry**. All features and models are **synthetic proxy constructs**.  
> No loss value or reconstruction threshold makes any food-safety claim.  
> See `docs/data_profile.md` and `docs/AD_LOG.md` (decisions D1, D9, D10, D11, D12).

---

### Colab Setup Instructions (Automated GitHub Workflow)

1. **Set up GitHub PAT in Colab Secrets**:
   - On the left toolbar in Google Colab, click the **Secrets** icon (the key 🔑).
   - Click **Add new secret**.
   - Name: `GITHUB_TOKEN` (or `GH_TOKEN`).
   - Value: Your GitHub Personal Access Token (classic or fine-grained with `repo` / contents write permissions).
   - Toggle **Notebook access** to **ON**.
2. **GPU Hardware Accelerator**:
   - Ensure a GPU runtime is selected: **Runtime > Change runtime type > Hardware accelerator > T4 GPU**.
3. **Automated Git Workflow**:
   - Running this notebook will automatically clone `https://github.com/Krishna200608/coldchain-ews-twin` using your PAT token.
   - The preprocessed window files (lstm_windows_{out,in}_{train,val}.npz) are automatically loaded directly from data/processed/ in the cloned repository.
   - Training executes on the Colab T4 GPU.
   - Once training completes, the notebook **automatically commits and pushes** the trained model weights (`models/lstm_out.h5`, `models/lstm_in.h5`) and training logs (`data/processed/lstm_training_log_{out,in}.json`) directly back to the GitHub repository's `main` branch!


In [ ]:
# 1. Authenticate with GitHub via Colab Secrets & Clone Repository
import os
import sys

# Retrieve GitHub PAT from Colab Secrets
try:
    from google.colab import userdata
    for token_key in ["GITHUB_TOKEN", "GH_TOKEN", "PAT"]:
        try:
            GITHUB_TOKEN = userdata.get(token_key)
            if GITHUB_TOKEN:
                print(f"Successfully retrieved GitHub token from secret key '{token_key}'.")
                break
        except Exception:
            continue
    else:
        raise ValueError("Could not find secret 'GITHUB_TOKEN' (or 'GH_TOKEN' / 'PAT').")
except Exception as e:
    raise RuntimeError(
        "Please add a secret named 'GITHUB_TOKEN' in Colab's Secrets tab (key icon on left sidebar) "
        "with your GitHub Personal Access Token and enable 'Notebook access'."
    ) from e

REPO_NAME = "coldchain-ews-twin"
REPO_OWNER = "Krishna200608"
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

# Clone or pull latest
if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_NAME} from GitHub...")
    !git clone {REPO_URL}
else:
    print(f"Repository {REPO_NAME} already exists. Pulling latest commits...")
    %cd {REPO_NAME}
    !git remote set-url origin {REPO_URL}
    !git pull
    %cd ..

# Enter repository root
%cd {REPO_NAME}
print(f"Current working directory: {os.getcwd()}")


In [ ]:
# 2. Hardware & GPU Environment Verification
import tensorflow as tf

print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("Physical GPUs Detected:", gpus)

# Enforce that training occurs on a GPU
assert len(gpus) > 0, "No GPU detected! Please navigate to Runtime > Change runtime type > T4 GPU."

# Display GPU hardware specifications via nvidia-smi
!nvidia-smi


In [ ]:
# 3. Data File Verification & Upload (if needed)
import pathlib
import numpy as np

pathlib.Path("models").mkdir(parents=True, exist_ok=True)
pathlib.Path("data/processed").mkdir(parents=True, exist_ok=True)

required_files = [
    "data/processed/lstm_windows_out_train.npz",
    "data/processed/lstm_windows_out_val.npz",
    "data/processed/lstm_windows_in_train.npz",
    "data/processed/lstm_windows_in_val.npz",
]

missing_files = []
for f in required_files:
    p = pathlib.Path(f)
    # Check if file was uploaded to root and needs moving
    if not p.exists() and pathlib.Path(p.name).exists():
        pathlib.Path(p.name).rename(p)
    if not p.exists():
        missing_files.append(f)

if missing_files:
    print(f"The following required window files are missing: {missing_files}")
    print("Please upload the .npz files using the upload button below:")
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded.keys():
        dest = pathlib.Path("data/processed") / fname
        pathlib.Path(fname).rename(dest)
        print(f"Moved {fname} -> {dest}")

# Verify all required files are present and inspect shapes
for f in required_files:
    p = pathlib.Path(f)
    assert p.exists(), f"Error: {f} is still missing. Please upload it."
    arr = np.load(p)["windows"]
    print(f"Verified {f:42s}: shape={arr.shape}, dtype={arr.dtype}")


In [ ]:
# 4. Model Architecture Definition (D12)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, RepeatVector, TimeDistributed, Dense
from tensorflow.keras.callbacks import EarlyStopping
import time
import json

LSTM_WINDOW_LENGTH = 30
N_FEATURES = 2
BATCH_SIZE = 256
MAX_EPOCHS = 50
PATIENCE = 5

def build_lstm_autoencoder(window_length=LSTM_WINDOW_LENGTH, n_features=N_FEATURES):
    """
    Builds the D12 LSTM-Autoencoder architecture:
    Encoder LSTM(32) -> Bottleneck -> RepeatVector(30) -> Decoder LSTM(32, return_sequences=True) -> TimeDistributed(Dense(2))
    Adam optimizer, MSE loss.
    """
    model = Sequential([
        LSTM(32, activation="tanh", input_shape=(window_length, n_features), name="encoder_lstm"),
        RepeatVector(window_length, name="bottleneck_repeat"),
        LSTM(32, activation="tanh", return_sequences=True, name="decoder_lstm"),
        TimeDistributed(Dense(n_features), name="reconstruction_dense")
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

# Inspect model architecture summary
sample_model = build_lstm_autoencoder()
sample_model.summary()


In [ ]:
# 5. Train Series Out LSTM-Autoencoder
X_out_train = np.load("data/processed/lstm_windows_out_train.npz")["windows"]
X_out_val   = np.load("data/processed/lstm_windows_out_val.npz")["windows"]

print(f"Training Series Out: Train shape={X_out_train.shape}, Val shape={X_out_val.shape}")

model_out = build_lstm_autoencoder()

early_stopping_out = EarlyStopping(
    monitor="val_loss",
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1
)

start_time_out = time.time()
history_out = model_out.fit(
    X_out_train, X_out_train,
    validation_data=(X_out_val, X_out_val),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping_out],
    verbose=1
)
elapsed_out = time.time() - start_time_out

# Save trained weights
model_out_path = "models/lstm_out.h5"
model_out.save(model_out_path)
print(f"Saved Series Out model to: {model_out_path}")

# Extract GPU hardware details
gpu_device_name = tf.test.gpu_device_name()
try:
    gpu_hardware = tf.config.experimental.get_device_details(tf.config.list_physical_devices('GPU')[0]).get('device_name', gpu_device_name)
except Exception:
    gpu_hardware = gpu_device_name

log_out = {
    "series": "Out",
    "architecture": "LSTM(32)-RepeatVector(30)-LSTM(32)-Dense(2)",
    "n_train_windows": len(X_out_train),
    "n_val_windows": len(X_out_val),
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "epochs_trained": len(history_out.history["loss"]),
    "final_train_loss": float(history_out.history["loss"][-1]),
    "final_val_loss": float(history_out.history["val_loss"][-1]),
    "best_val_loss": float(min(history_out.history["val_loss"])),
    "training_wall_clock_sec": round(elapsed_out, 2),
    "gpu_device": str(gpu_hardware),
}

with open("data/processed/lstm_training_log_out.json", "w", encoding="utf-8") as f:
    json.dump(log_out, f, indent=2)
print("Saved training log: data/processed/lstm_training_log_out.json")


In [ ]:
# 7. Loss Curves & Training Diagnostics
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, hist, s_name in zip(axes, [history_out, history_in], ["Out", "In"]):
    ax.plot(hist.history["loss"], label="Train Loss (MSE)", color="#2b8cbe", linewidth=2)
    ax.plot(hist.history["val_loss"], label="Val Loss (MSE)", color="#e41a1c", linewidth=2)
    ax.set_title(f"Reconstruction Loss: Series '{s_name}'", fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.legend(frameon=True)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Training Run Summary:")
print("Series Out:", log_out)
print("Series In :", log_in)


In [ ]:
# 8. Commit & Push Trained Models and Logs Directly to GitHub
import os

print("Configuring git user credentials...")
!git config user.name "Krishna200608"
!git config user.email "krishnasikheriya001@gmail.com"

# Ensure origin remote is authenticated with PAT
!git remote set-url origin https://{GITHUB_TOKEN}@github.com/Krishna200608/coldchain-ews-twin.git

# Stage trained model weights and logs (using -f to override .gitignore for these specific artifacts)
!git add -f models/lstm_out.h5 models/lstm_in.h5 data/processed/lstm_training_log_out.json data/processed/lstm_training_log_in.json

# Check staged changes
!git status

# Commit and push
!git commit -m "feat(colab): add trained LSTM-Autoencoder weights and training logs from Colab T4 GPU"
!git push origin main

print("\nSuccessfully pushed trained models and training logs to GitHub origin/main!")


### 9. Verification & Next Steps
Once the push succeeds from Colab, the GitHub repository will contain:
- `models/lstm_out.h5`
- `models/lstm_in.h5`
- `data/processed/lstm_training_log_out.json`
- `data/processed/lstm_training_log_in.json`

On your local machine, run:
```bash
git pull origin main
```
to synchronize the trained models and training logs locally.
Then proceed to Milestone 4b (evaluating reconstruction error on test-side windows and benchmarking early warning lead times against Isolation Forest and the Naive Baseline).
